# 07 — Vasicek Model: Monte Carlo Simulation

Spec §7.4–7.5. Depends on calibrated parameters from notebook 06.

Pipeline:
1. Load calibrated parameters (κ, θ, σ, r₀) from FRED cache
2. Simulate 5 000 paths × 360 monthly steps (30 years)
3. Fan plot — short rate paths
4. Verify terminal distribution matches N(θ, σ²/2κ) by KS test
5. Yield curve evolution across simulation horizon
6. Convert short rate paths → 30Y mortgage rate paths
7. Mortgage rate fan plot and percentile statistics

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from scipy import stats

load_dotenv()
sys.path.insert(0, str(Path('..') / 'src'))
import vasicek as v

FRED_DIR = Path('..') / 'data' / 'fred'
FIGURES  = Path('..') / 'artifacts' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

N_PATHS  = 5_000
N_STEPS  = 360      # 30 years monthly
TIME_STEP = 1.0 / 12

## 1. Load calibrated parameters

In [ ]:
kappa, theta, sigma, initial_rate = v.load_and_calibrate(
    start='2000-01-01',
    end='2025-12-31',
    cache_path=FRED_DIR / 'gs1_monthly.csv',
)
lr_mean, lr_std, _ = v.long_run_distribution(kappa, theta, sigma)

print(f'κ={kappa:.4f}  θ={theta:.4f}  σ={sigma:.6f}  r₀={initial_rate:.4f}')
print(f'Stationary: mean={lr_mean*100:.2f}%  std={lr_std*100:.2f}%')
print(f'Simulation: {N_PATHS:,} paths × {N_STEPS} steps ({N_STEPS//12} years monthly)')

## 2. Simulate 5 000 short rate paths

In [ ]:
rate_paths = v.simulate_vasicek(
    initial_rate=initial_rate,
    kappa=kappa,
    theta=theta,
    sigma=sigma,
    n_paths=N_PATHS,
    n_steps=N_STEPS,
    time_step=TIME_STEP,
    seed=42,
)
print(f'Paths shape: {rate_paths.shape}  (n_paths × n_steps+1)')
print(f'Column 0 all equal r₀: {np.allclose(rate_paths[:, 0], initial_rate)}')

months = np.arange(N_STEPS + 1)

## 3. Fan plot — short rate paths

In [ ]:
pct_5   = np.percentile(rate_paths, 5,  axis=0)
pct_25  = np.percentile(rate_paths, 25, axis=0)
pct_50  = np.percentile(rate_paths, 50, axis=0)
pct_75  = np.percentile(rate_paths, 75, axis=0)
pct_95  = np.percentile(rate_paths, 95, axis=0)

fig, ax = plt.subplots(figsize=(14, 6))

# Sample paths (50 for visibility)
rng = np.random.default_rng(0)
sample_idx = rng.choice(N_PATHS, size=50, replace=False)
for idx in sample_idx:
    ax.plot(months / 12, rate_paths[idx] * 100, color='steelblue',
            alpha=0.08, linewidth=0.5)

# Percentile bands
ax.fill_between(months / 12, pct_5 * 100, pct_95 * 100,
                alpha=0.15, color='steelblue', label='5–95th pct')
ax.fill_between(months / 12, pct_25 * 100, pct_75 * 100,
                alpha=0.25, color='steelblue', label='25–75th pct')
ax.plot(months / 12, pct_50 * 100, 'b-', linewidth=2, label='Median')
ax.axhline(theta * 100, color='red', linestyle='--', linewidth=1.5,
           label=f'θ = {theta*100:.2f}%')

ax.set_title(f'Vasicek Short Rate Simulation: {N_PATHS:,} Paths × 30 Years')
ax.set_xlabel('Years')
ax.set_ylabel('Short Rate (%)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES / 'vasicek_rate_fan.png', bbox_inches='tight')
plt.show()

print('Percentiles at 30Y horizon:')
for p, val in [(5, pct_5[-1]), (25, pct_25[-1]), (50, pct_50[-1]),
               (75, pct_75[-1]), (95, pct_95[-1])]:
    print(f'  P{p:2d}: {val*100:.2f}%')

## 4. Verify terminal distribution (KS test against theoretical)

At 30 years, the distribution of r_T should be approximately N(θ, σ²/2κ)  
since the transient term decays as exp(-κ·30) ≈ 0.

In [ ]:
terminal = rate_paths[:, -1]

# KS test against N(θ, σ²/2κ)
ks_stat, ks_p = stats.kstest(
    terminal,
    'norm',
    args=(lr_mean, lr_std),
)

print(f'Terminal distribution (t=30Y, {N_PATHS:,} paths):')
print(f'  Simulated mean:  {terminal.mean()*100:.4f}%  (theoretical: {lr_mean*100:.4f}%)')
print(f'  Simulated std:   {terminal.std()*100:.4f}%   (theoretical: {lr_std*100:.4f}%)')
print(f'  KS statistic:    {ks_stat:.4f}')
print(f'  KS p-value:      {ks_p:.4f}  (expect > 0.05 for good fit)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram vs. theoretical density
r_grid = np.linspace(terminal.min(), terminal.max(), 200)
theory_pdf = stats.norm.pdf(r_grid, lr_mean, lr_std)
axes[0].hist(terminal * 100, bins=60, density=True, color='steelblue',
             alpha=0.6, label='Simulated')
axes[0].plot(r_grid * 100, theory_pdf / 100, 'r-', linewidth=2,
             label=f'N(θ={lr_mean*100:.2f}%, σ_∞={lr_std*100:.2f}%)')
axes[0].set_title(f'Terminal Distribution at t=30Y (KS p={ks_p:.3f})')
axes[0].set_xlabel('Short Rate (%)')
axes[0].set_ylabel('Density')
axes[0].legend()

# QQ plot
stats.probplot(terminal, dist='norm', sparams=(lr_mean, lr_std), plot=axes[1])
axes[1].set_title('QQ Plot vs. N(θ, σ²/2κ)')

plt.tight_layout()
plt.savefig(FIGURES / 'vasicek_terminal_distribution.png', bbox_inches='tight')
plt.show()

## 5. Yield curve evolution across simulation horizon

In [ ]:
maturities = [0.25, 1, 2, 5, 10, 30]
snapshot_months = [0, 60, 120, 240, 360]

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(snapshot_months)))

for snap_month, color in zip(snapshot_months, colors):
    # Median short rate at this horizon
    r_median = np.median(rate_paths[:, snap_month])
    yc = v.vasicek_yield_curve(r_median, kappa, theta, sigma, maturities=maturities)
    ax.plot(yc['maturity_yrs'], yc['yield_pct'], 'o-',
            color=color, linewidth=1.8, markersize=5,
            label=f't={snap_month//12}Y  (r_median={r_median*100:.2f}%)')

ax.axhline(theta * 100, color='gray', linestyle=':', alpha=0.8,
           label=f'θ = {theta*100:.2f}%')
ax.set_title('Vasicek Yield Curve Evolution\n(median short rate at each horizon)')
ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Yield (%)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES / 'vasicek_yield_curve_evolution.png', bbox_inches='tight')
plt.show()

## 6. Convert to 30-year mortgage rate paths (spec §7.5)

Mortgage rate = Vasicek 30Y yield + MBS-Treasury spread (150bp).

In [ ]:
mortgage_paths = v.mortgage_rate_from_short(
    rate_paths, kappa, theta, sigma, tau=30.0, mbs_spread=0.015
)
print(f'Mortgage rate paths shape: {mortgage_paths.shape}')
print(f'Current 30Y mortgage rate (t=0): {mortgage_paths[:, 0].mean()*100:.2f}%')
print(f'30Y horizon median mortgage rate: {np.median(mortgage_paths[:, -1])*100:.2f}%')
print(f'30Y horizon range P5–P95: {np.percentile(mortgage_paths[:,-1],5)*100:.2f}% – '
      f'{np.percentile(mortgage_paths[:,-1],95)*100:.2f}%')

## 7. Mortgage rate fan plot

In [ ]:
mp5  = np.percentile(mortgage_paths, 5,  axis=0)
mp25 = np.percentile(mortgage_paths, 25, axis=0)
mp50 = np.percentile(mortgage_paths, 50, axis=0)
mp75 = np.percentile(mortgage_paths, 75, axis=0)
mp95 = np.percentile(mortgage_paths, 95, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: fan plot
for idx in sample_idx:
    axes[0].plot(months / 12, mortgage_paths[idx] * 100,
                 color='darkorange', alpha=0.06, linewidth=0.5)
axes[0].fill_between(months / 12, mp5 * 100, mp95 * 100,
                     alpha=0.15, color='darkorange', label='5–95th pct')
axes[0].fill_between(months / 12, mp25 * 100, mp75 * 100,
                     alpha=0.25, color='darkorange', label='25–75th pct')
axes[0].plot(months / 12, mp50 * 100, 'darkorange', linewidth=2, label='Median')
axes[0].set_title(f'30Y Mortgage Rate Paths ({N_PATHS:,} simulations)')
axes[0].set_xlabel('Years')
axes[0].set_ylabel('30Y Mortgage Rate (%)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: terminal distribution of mortgage rates
axes[1].hist(mortgage_paths[:, -1] * 100, bins=60, density=True,
             color='darkorange', alpha=0.7)
axes[1].axvline(mp50[-1] * 100, color='k', linestyle='--',
                label=f'Median {mp50[-1]*100:.2f}%')
axes[1].axvline(mp5[-1] * 100, color='gray', linestyle=':',
                label=f'P5 {mp5[-1]*100:.2f}%')
axes[1].axvline(mp95[-1] * 100, color='gray', linestyle=':',
                label=f'P95 {mp95[-1]*100:.2f}%')
axes[1].set_title('Terminal 30Y Mortgage Rate Distribution (t=30Y)')
axes[1].set_xlabel('30Y Mortgage Rate (%)')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES / 'vasicek_mortgage_rate_fan.png', bbox_inches='tight')
plt.show()

print('\nMortgage rate percentiles at key horizons:')
header = f'{"Horizon":>10}  {"P5":>8}  {"P25":>8}  {"P50":>8}  {"P75":>8}  {"P95":>8}'
print(header)
for months_h in [12, 60, 120, 240, 360]:
    p5, p25, p50, p75, p95 = [np.percentile(mortgage_paths[:, months_h], pct) * 100
                               for pct in [5, 25, 50, 75, 95]]
    print(f'{months_h//12}Y{"":>7}  {p5:8.2f}  {p25:8.2f}  {p50:8.2f}  {p75:8.2f}  {p95:8.2f}')